# ML Prediction API — FastAPI App
This notebook walks through building, training, and serving the ML Prediction API defined in `app.py`.

## 1. Install Dependencies

In [10]:
!pip install fastapi uvicorn joblib scikit-learn numpy pydantic

## 2. Train & Save a Sample Model

In [11]:
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression

# Generate dummy training data with 4 features
np.random.seed(42)
X_train = np.random.rand(100, 4)
y_train = X_train @ np.array([1.5, -2.0, 3.0, 0.5]) + np.random.randn(100) * 0.1

model = LinearRegression()
model.fit(X_train, y_train)

joblib.dump(model, 'model.joblib')
print(f'Model trained. n_features_in_: {model.n_features_in_}')
print(f'Coefficients: {model.coef_}')

Model trained. n_features_in_: 4
Coefficients: [ 1.54405179 -2.06072169  3.01471804  0.47559898]


## 3. Pydantic Schemas

In [12]:
from typing import List
from pydantic import BaseModel, ConfigDict, Field

class InputData(BaseModel):
    features: List[float] = Field(..., description='Input features for prediction')
    model_config = ConfigDict(json_schema_extra={'example': {'features': [1.0, 2.0, 3.0, 4.0]}})

class OutputData(BaseModel):
    prediction: float = Field(..., description='Model prediction')

# Quick validation test
sample = InputData(features=[1.0, 2.0, 3.0, 4.0])
print('Parsed input:', sample)

Parsed input: features=[1.0, 2.0, 3.0, 4.0]


## 4. FastAPI App Definition (`app.py`)

In [13]:
%%writefile app.py
"""
app.py
------
A FastAPI application that exposes a trained machine learning model as a
REST API. Clients send a list of numerical features via HTTP POST and receive
a predicted value in return.
"""

import logging
from pathlib import Path
from typing import List

import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, Field

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class InputData(BaseModel):
    features: List[float] = Field(..., description='Input features for prediction')
    model_config = ConfigDict(json_schema_extra={'example': {'features': [1.0, 2.0, 3.0, 4.0]}})

class OutputData(BaseModel):
    prediction: float = Field(..., description='Model prediction')

app = FastAPI(title='ML Prediction API', version='1.0.0')

MODEL_PATH = Path(__file__).parent / 'model.joblib'

try:
    model = joblib.load(str(MODEL_PATH))
    logger.info(f'Model loaded from {MODEL_PATH}')
    EXPECTED_FEATURES = model.n_features_in_
except Exception as e:
    logger.error(f'Failed to load model: {e}')
    model = None
    EXPECTED_FEATURES = 4

@app.post('/predict', response_model=OutputData)
async def predict(data: InputData) -> OutputData:
    try:
        if len(data.features) != EXPECTED_FEATURES:
            raise HTTPException(
                status_code=422,
                detail=f'Expected {EXPECTED_FEATURES} features, got {len(data.features)}',
            )
        if model is None:
            raise HTTPException(status_code=500, detail='Model not loaded')

        X = np.array(data.features).reshape(1, -1)
        prediction = model.predict(X)[0]
        return OutputData(prediction=float(prediction))

    except HTTPException:
        raise
    except Exception as e:
        logger.error(f'Prediction error: {e}')
        raise HTTPException(status_code=500, detail=f'Model inference failed: {str(e)}') from e

Overwriting app.py


## 5. Run the Server
Start the server in the background (for notebook testing).

In [14]:
import subprocess, time, requests

server = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait until the server is actually accepting connections (up to 15 s)
for i in range(30):
    try:
        requests.get('http://127.0.0.1:8000/docs', timeout=1)
        print(f'Server ready after ~{(i+1)*0.5:.1f}s  (PID: {server.pid})')
        break
    except requests.exceptions.ConnectionError:
        time.sleep(0.5)
else:
    print('WARNING: server did not become ready in time — check for port conflicts or import errors.')
    print(server.stderr.read().decode())

Server ready after ~2.0s  (PID: 11700)


## 6. Test the `/predict` Endpoint

In [15]:
import requests

BASE_URL = 'http://127.0.0.1:8000'

# --- Happy path ---
payload = {'features': [0.5, 0.3, 0.8, 0.1]}
response = requests.post(f'{BASE_URL}/predict', json=payload)
print('Status:', response.status_code)
print('Response:', response.json())

Status: 200
Response: {'prediction': 2.6295248847516657}


In [16]:
# --- Wrong number of features (expect 422) ---
bad_payload = {'features': [1.0, 2.0]}
r = requests.post(f'{BASE_URL}/predict', json=bad_payload)
print('Status:', r.status_code)
print('Detail:', r.json())

Status: 422
Detail: {'detail': 'Expected 4 features, got 2'}


## 7. Direct Inference (Without HTTP Server)

In [17]:
loaded_model = joblib.load('model.joblib')

features = [0.5, 0.3, 0.8, 0.1]
X = np.array(features).reshape(1, -1)
pred = loaded_model.predict(X)[0]
print(f'Direct prediction: {pred:.4f}')

Direct prediction: 2.6295


## 8. Shutdown the Server

In [18]:
server.terminate()
print('Server stopped.')

Server stopped.
